In [14]:
import pandas as pd

INPUT_CSV = "bayes_activity_data.csv"

# Load dataset
df = pd.read_csv(INPUT_CSV)

# Filter only jogging rows
jogging_df = df[df['activity_name'].str.lower() == 'jogging']

# Select numerical features
feature_columns = [
    'mean_x', 'mean_y', 'mean_z',
    'pos_count_x', 'pos_count_y', 'pos_count_z',
    'fft_std_dev_x', 'fft_std_dev_y', 'fft_std_dev_z',
    'signal_magnitude_area'
]

# Describe jogging data
description = jogging_df[feature_columns].describe()

print("=== Jogging Data Statistics ===\n")
print(description)


=== Jogging Data Statistics ===

            mean_x       mean_y       mean_z  pos_count_x  pos_count_y  \
count  2138.000000  2138.000000  2138.000000  2138.000000  2138.000000   
mean     -0.420837     5.562568    -0.246008    58.702526    88.688962   
std       5.099923     4.242662     1.261220    25.518368    21.176180   
min      -9.836172   -10.818281    -4.283047     0.000000     2.000000   
25%      -4.728477     3.705195    -1.105723    38.000000    78.000000   
50%      -1.208086     7.156602    -0.304089    55.000000    95.000000   
75%       4.148496     8.308223     0.662109    82.000000   104.000000   
max       9.353985    10.790827     3.795859   120.000000   128.000000   

       pos_count_z  fft_std_dev_x  fft_std_dev_y  fft_std_dev_z  \
count  2138.000000    2138.000000    2138.000000    2138.000000   
mean     55.222170      79.952849      99.169765      46.710132   
std      16.034349      25.558793      23.417523      11.800699   
min       6.000000      23.04887

-----------------------------------------------------------------------------------------

In [31]:
import pandas as pd
import numpy as np

# ==========================================
# CONFIGURATION
# ==========================================
# Path to your REAL RAW WISDM data file
RAW_DATA_PATH = "WISDM_ar_v1.1_raw.txt" 
OUTPUT_CSV = "real_data_features_c_compatible.csv"

# MUST match the window size you send from Python to C (e.g., 128)
WINDOW_SIZE = 128 
STEP_SIZE = 64 

def clean_wisdm_data(file_path):
    """Reads the raw text file and fixes formatting issues."""
    print(f"Reading raw data from {file_path}...")
    columns = ['user', 'activity', 'timestamp', 'x', 'y', 'z']
    data = []
    bad_lines = 0
    
    with open(file_path, 'r') as f:
        for line in f:
            # Normalize and split
            parts = [p.strip() for p in line.strip().replace(';', '').split(',')]
            if len(parts) != 6:
                continue
            
            # Check numeric fields and convert, skip if malformed or empty
            x_str, y_str, z_str = parts[3], parts[4], parts[5]
            try:
                x = float(x_str)
                y = float(y_str)
                z = float(z_str)
            except (ValueError, TypeError):
                bad_lines += 1
                continue
            
            # Append converted numeric values to avoid later astype() errors
            data.append([parts[0], parts[1], parts[2], x, y, z])
    
    if bad_lines:
        print(f"⚠️ Skipped {bad_lines} malformed or empty lines while parsing.")
    
    df = pd.DataFrame(data, columns=columns)
    # Ensure the numeric columns are typed correctly
    df['x'] = df['x'].astype(float)
    df['y'] = df['y'].astype(float)
    df['z'] = df['z'].astype(float)
    return df

def extract_features_for_c(window):
    """
    Calculates features EXACTLY how your C code does.
    """
    xs, ys, zs = window['x'].values, window['y'].values, window['z'].values
    
    features = {}
    
    # 1. Mean
    features['mean_x'] = np.mean(xs)
    features['mean_y'] = np.mean(ys)
    features['mean_z'] = np.mean(zs)
    
    # 2. Pos Count (Values > 0)
    features['pos_count_x'] = np.sum(xs > 0)
    features['pos_count_y'] = np.sum(ys > 0)
    features['pos_count_z'] = np.sum(zs > 0)
    
    # 3. Standard Deviation (Time Domain)
    # YOUR C CODE uses simple std dev. We must match that here.
    # (The original dataset called this 'fft_std_dev', but we use simple std)
    features['fft_std_dev_x'] = np.std(xs)
    features['fft_std_dev_y'] = np.std(ys)
    features['fft_std_dev_z'] = np.std(zs)
    
    # 4. SMA
    features['signal_magnitude_area'] = (np.sum(np.abs(xs)) + np.sum(np.abs(ys)) + np.sum(np.abs(zs))) / len(xs)
    
    return features

# --- Main Execution ---
df = clean_wisdm_data(RAW_DATA_PATH)

print(f"Processing {len(df)} raw samples...")
processed_data = []

# Encode Activities to 0-5
activities = sorted(df['activity'].unique())
act_map = {act: i for i, act in enumerate(activities)}
print(f"Class Mapping: {act_map}")

for i in range(0, len(df) - WINDOW_SIZE, STEP_SIZE):
    window = df.iloc[i : i + WINDOW_SIZE]
    
    # Only use windows with one activity type
    if window['activity'].nunique() == 1:
        act_name = window['activity'].iloc[0]
        
        feats = extract_features_for_c(window)
        feats['activity_code'] = act_map[act_name]
        feats['activity_name'] = act_name
        processed_data.append(feats)

# Save
out_df = pd.DataFrame(processed_data)
out_df.to_csv(OUTPUT_CSV, index=False)
print(f"✅ Saved {len(out_df)} samples to {OUTPUT_CSV}")
print("Now train your model on THIS file.")

Reading raw data from WISDM_ar_v1.1_raw.txt...
⚠️ Skipped 1 malformed or empty lines while parsing.
Processing 1086465 raw samples...
Class Mapping: {'Downstairs': 0, 'Jogging': 1, 'Sitting': 2, 'Standing': 3, 'Upstairs': 4, 'Walking': 5}
✅ Saved 16210 samples to real_data_features_c_compatible.csv
Now train your model on THIS file.


In [30]:
import pandas as pd
import numpy as np
import os
from sklearn2c import BayesClassifier

# Input the file created in Step 1
CSV_FILE = "real_data_features_c_compatible.csv"
EXPORT_NAME = "bayes_cls_config"

def train():
    print(f"Loading {CSV_FILE}...")
    df = pd.read_csv(CSV_FILE)
    
    # Features expected by C code
    feature_cols = [
        'mean_x', 'mean_y', 'mean_z',
        'pos_count_x', 'pos_count_y', 'pos_count_z',
        'fft_std_dev_x', 'fft_std_dev_y', 'fft_std_dev_z',
        'signal_magnitude_area'
    ]
    
    X = df[feature_cols].values.astype(np.float32)
    y = df['activity_code'].values.astype(int)
    
    print(f"Training Bayes on {len(X)} real samples...")
    clf = BayesClassifier()
    clf.train(X, y)
    
    print("Exporting C config...")
    clf.export(EXPORT_NAME)
    print(f"✅ Done! Replace {EXPORT_NAME}.h and .c in your Keil project.")

if __name__ == "__main__":
    train()

Loading real_data_features_c_compatible.csv...
Training Bayes on 16210 real samples...
Exporting C config...
✅ Done! Replace bayes_cls_config.h and .c in your Keil project.
